In [ ]:
import pandas as pd
from pathlib import Path

pasta = Path("../data/raw/microdados_enem_2025")

arquivos ={
    "participantes": pasta / "DADOS" / "PARTICIPANTES_2025.csv",
    "resultados": pasta / "DADOS" / "RESULTADOS_2025.csv",
}

for nome, caminho in  arquivos.items():
    amostra = pd.read_csv(caminho, sep=";", encoding="latin-1", nrows=5)
    print(nome, amostra.shape)
    print(list(amostra.columns))
    print()



In [ ]:
achados = list(pasta.glob("*/*.xlsx"))
dic = pd.read_excel(achados[0], sheet_name=None)

for aba, tabela in dic.items():
    print(aba, tabela.shape)

In [ ]:
res = dic["RESULTADOS_2025"]
print(res.columns.tolist())
print(res.head(8).to_string())


In [ ]:
def buscar(tabela, texto):
    achou = tabela.astype(str).apply(
        lambda coluna: coluna.str.contains(texto, case=False, na=False)
    ).any(axis=1)
    return tabela[achou]

print(buscar(res, "NOTA").to_string())

In [ ]:
def mostrar(tabela, variavel, linhas=8):
    posicoes = tabela.index[tabela.iloc[:, 0] == variavel]
    for p in posicoes:
        print(tabela.iloc[p : p + linhas].to_string(header=False))
        print()


for v in ["TP_PRESENCA_CN", "TP_STATUS_REDACAO", "TP_DEPENDENCIA_ADM_ESC",
          "TP_LOCALIZACAO_ESC", "SG_UF_ESC", "SG_UF_PROVA", "CO_ESCOLA"]:
    print("=====", v)
    mostrar(res, v)

print("===== RODAPÉ")
print(res.tail(15).to_string(header=False))

In [ ]:
colunas = [
    "SG_UF_ESC", "SG_UF_PROVA", "TP_DEPENDENCIA_ADM_ESC", "TP_LOCALIZACAO_ESC",
    "TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT",
    "NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT",
    "TP_STATUS_REDACAO", "NU_NOTA_REDACAO",
]

df = pd.read_csv(arquivos["resultados"], sep=";", encoding="latin-1", usecols=colunas)

print(df.shape)
print(df.dtypes)
print(df.head())

In [ ]:
areas = ["CN", "CH", "LC", "MT"]

for area in areas:
    print("Presença", area)
    print(df[f"TP_PRESENCA_{area}"].value_counts().sort_index())
    print()

print("Status da redação")
print(df["TP_STATUS_REDACAO"].value_counts(dropna=False).sort_index())
print()

print("Proporção sem escola informada:", df["TP_DEPENDENCIA_ADM_ESC"].isna().mean())
print("Linhas sem UF da prova:", df["SG_UF_PROVA"].isna().sum())
print()

for area in areas:
    sem_presenca = df[f"TP_PRESENCA_{area}"] != 1
    com_nota = df[f"NU_NOTA_{area}"].notna()
    print(area, "- linhas com nota e presença diferente de 1:", (sem_presenca & com_nota).sum())

In [ ]:

print(
    df.groupby("TP_STATUS_REDACAO")["NU_NOTA_REDACAO"]
    .agg(["count", "mean", "min", "max"])
    .round(1)
)
print()


so_status_1 = df[df["TP_STATUS_REDACAO"] == 1]["NU_NOTA_REDACAO"]
todas_com_status = df[df["TP_STATUS_REDACAO"].notna()]["NU_NOTA_REDACAO"]
print("Redação, só status 1:", round(so_status_1.mean(), 1))
print("Redação, todas com status (zeros incluídos):", round(todas_com_status.mean(), 1))
print()


for area in areas:
    presentes = df[df[f"TP_PRESENCA_{area}"] == 1]
    media = round(presentes[f"NU_NOTA_{area}"].mean(), 1)
    vazias = presentes[f"NU_NOTA_{area}"].isna().sum()
    print(area, "| presentes:", len(presentes), "| média:", media, "| notas vazias:", vazias)
print()


presentes_lc = df[df["TP_PRESENCA_LC"] == 1]
print("Com escola informada:", round(presentes_lc["TP_DEPENDENCIA_ADM_ESC"].notna().mean(), 3))
print(presentes_lc["TP_DEPENDENCIA_ADM_ESC"].value_counts(normalize=True).sort_index().round(3))
print(
    "UF da escola vazia, mas dependência preenchida:",
    (df["SG_UF_ESC"].isna() & df["TP_DEPENDENCIA_ADM_ESC"].notna()).sum(),
)

In [ ]:
UF_REGIAO = {
    "AC": "Norte", "AM": "Norte", "AP": "Norte", "PA": "Norte",
    "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste",
    "PB": "Nordeste", "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste",
    "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MS": "Centro-Oeste",
    "MT": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}

print("UFs que não estão no dicionário:", set(df["SG_UF_PROVA"].unique()) - set(UF_REGIAO))

df["REGIAO"] = df["SG_UF_PROVA"].map(UF_REGIAO)


def tabela_medias(dados, coluna_grupo):
    colunas = {}
    for area in areas:
        presentes = dados[dados[f"TP_PRESENCA_{area}"] == 1]
        colunas[area] = presentes.groupby(coluna_grupo)[f"NU_NOTA_{area}"].mean()
    com_status = dados[dados["TP_STATUS_REDACAO"].notna()]
    colunas["RED_com_zeros"] = com_status.groupby(coluna_grupo)["NU_NOTA_REDACAO"].mean()
    validas = dados[dados["TP_STATUS_REDACAO"] == 1]
    colunas["RED_status1"] = validas.groupby(coluna_grupo)["NU_NOTA_REDACAO"].mean()
    colunas["n_LC"] = dados[dados["TP_PRESENCA_LC"] == 1].groupby(coluna_grupo).size()
    colunas["n_CN"] = dados[dados["TP_PRESENCA_CN"] == 1].groupby(coluna_grupo).size()
    return pd.DataFrame(colunas)


medias_regiao = tabela_medias(df, "REGIAO")
medias_uf = tabela_medias(df, "SG_UF_PROVA")

print(medias_regiao.round(1).sort_values("LC", ascending=False).to_string())
print()
print(medias_uf.round(1).sort_values("LC", ascending=False).to_string())

In [ ]:
faltas = pd.DataFrame({
    "faltou_LC_%": df.groupby("SG_UF_PROVA")["TP_PRESENCA_LC"].apply(lambda s: (s == 0).mean() * 100),
    "faltou_CN_%": df.groupby("SG_UF_PROVA")["TP_PRESENCA_CN"].apply(lambda s: (s == 0).mean() * 100),
})

resumo = medias_uf[["LC", "CN"]].join(faltas).round(1)
print(resumo.sort_values("faltou_LC_%", ascending=False).to_string())
print()
print(resumo.corr().round(2))
print()

lc = df[df["TP_PRESENCA_LC"] == 1]
dispersao = lc.groupby("SG_UF_PROVA")["NU_NOTA_LC"].agg(["mean", "median", "std"]).round(1)
print(dispersao.sort_values("mean", ascending=False).to_string())

In [ ]:
com_escola = df[df["TP_DEPENDENCIA_ADM_ESC"].notna()].copy()

nomes_dep = {1: "Federal", 2: "Estadual", 3: "Municipal", 4: "Privada"}
nomes_rede = {1: "Pública", 2: "Pública", 3: "Pública", 4: "Privada"}
com_escola["DEPENDENCIA"] = com_escola["TP_DEPENDENCIA_ADM_ESC"].map(nomes_dep)
com_escola["REDE"] = com_escola["TP_DEPENDENCIA_ADM_ESC"].map(nomes_rede)

medias_dep = tabela_medias(com_escola, "DEPENDENCIA")
medias_rede = tabela_medias(com_escola, "REDE")

print(medias_dep.round(1).to_string())
print()
print(medias_rede.round(1).to_string())
print()

cols_notas = ["CN", "CH", "LC", "MT", "RED_com_zeros", "RED_status1"]
diferenca = medias_rede.loc["Privada", cols_notas] - medias_rede.loc["Pública", cols_notas]
print("Privada menos Pública:")
print(diferenca.round(1))
print()

presentes_lc = df[df["TP_PRESENCA_LC"] == 1]
cobertura = presentes_lc.groupby("SG_UF_PROVA")["TP_DEPENDENCIA_ADM_ESC"].apply(
    lambda s: s.notna().mean() * 100
)
print("% de presentes em LC com escola informada, por estado:")
print(cobertura.round(1).sort_values().to_string())

In [ ]:
com_escola["REGIAO_ESC"] = com_escola["SG_UF_ESC"].map(UF_REGIAO)


def pivo(dados, coluna_grupo, area):
    presentes = dados[dados[f"TP_PRESENCA_{area}"] == 1]
    nota = f"NU_NOTA_{area}"
    medias = presentes.pivot_table(index=coluna_grupo, columns="REDE", values=nota, aggfunc="mean")
    contagens = presentes.pivot_table(index=coluna_grupo, columns="REDE", values=nota, aggfunc="count")
    return pd.DataFrame({
        "publica": medias["Pública"],
        "privada": medias["Privada"],
        "gap": medias["Privada"] - medias["Pública"],
        "n_pub": contagens["Pública"],
        "n_priv": contagens["Privada"],
    })


for area in ["LC", "MT"]:
    print("=====", area, "por região da escola")
    print(pivo(com_escola, "REGIAO_ESC", area).round(1).to_string())
    print()
    por_uf = pivo(com_escola, "SG_UF_ESC", area)
    print(area, "- estados com menos de 500 alunos de escola privada (excluídos):",
          list(por_uf.index[por_uf["n_priv"] < 500]))
    por_uf = por_uf[por_uf["n_priv"] >= 500]
    print(por_uf.round(1).sort_values("gap").to_string())
    print()

print("===== Diferença Privada - Pública em desvios-padrão")
for area in areas:
    notas = com_escola[com_escola[f"TP_PRESENCA_{area}"] == 1][f"NU_NOTA_{area}"]
    print(area, round(diferenca[area] / notas.std(), 2))
com_status = com_escola[com_escola["TP_STATUS_REDACAO"].notna()]["NU_NOTA_REDACAO"]
valida = com_escola[com_escola["TP_STATUS_REDACAO"] == 1]["NU_NOTA_REDACAO"]
print("RED_com_zeros", round(diferenca["RED_com_zeros"] / com_status.std(), 2))
print("RED_status1", round(diferenca["RED_status1"] / valida.std(), 2))

In [ ]:
from pypdf import PdfReader

pdf_read_me = list(pasta.glob("*/Leia_Me_Enem_2025.pdf"))
print(pdf_read_me)

read = PdfReader(pdf_read_me[0])
for i, pagina in enumerate(read.pages):
    texto = pagina.extract_text() or ""
    for linha in texto.split("\n"):
        if "escola" in linha.lower():
            print(i + 1, "|", linha)

In [ ]:
import matplotlib.pyplot as plt

for area in areas:
    notas = com_escola.loc[com_escola[f"TP_PRESENCA_{area}"] == 1, f"NU_NOTA_{area}"]
    print(area, "desvio-padrão:", round(notas.std(), 1))

Path("../images").mkdir(exist_ok=True)

fig, eixos = plt.subplots(1, 2, figsize=(15, 8))

# Gráfico 1: qual UF tem a maior média de LC?
ordem = medias_uf["LC"].sort_values()
eixos[0].scatter(ordem.values, ordem.index)
media_nacional_lc = df.loc[df["TP_PRESENCA_LC"] == 1, "NU_NOTA_LC"].mean()
eixos[0].axvline(media_nacional_lc, linestyle="--", color="gray")
eixos[0].set_title("Média de LC por UF da prova (presentes)")
eixos[0].set_xlabel("Nota")

# Gráfico 2: a diferença entre redes se repete em todas as regiões?
reg = pivo(com_escola, "REGIAO_ESC", "LC")[["publica", "privada"]]
reg.plot(kind="bar", ax=eixos[1])
eixos[1].set_title("Média de LC por região da escola e rede")
eixos[1].set_ylabel("Nota")
eixos[1].set_xlabel("")

plt.tight_layout()
fig.savefig("../images/lc_estados_e_redes.png", dpi=150)
plt.show()

In [ ]:
for numero in [7, 8]:
    print("=== página", numero)
    print(read.pages[numero - 1].extract_text())

In [ ]:
reg = pivo(com_escola, "REGIAO_ESC", "LC")[["publica", "privada"]]
reg.columns = ["Public", "Private"]
reg.plot(kind="bar", ax=eixos[1])
eixos[1].set_title("Mean LC score by school region and school type\n(only participants with school information)")
eixos[1].set_ylabel("Score")
eixos[1].set_xlabel("")
eixos[1].legend(loc="lower right")
eixos[1].tick_params(axis="x", rotation=0)

### Chart notes

**Mean LC score by state:** the eight highest means (SP, DF, SC, RS, RJ, PR, MG, ES) are separated from the rest by a gap of about 9 points; almost all Northeast and North states fall below the national mean. The spread between the highest and the lowest state (about 61 points) is smaller than the standard deviation of individual scores (about 73 points).

**Public vs. private (LC):** the private-school mean is higher in every region, with the smallest gaps in the South and Southeast. This covers only participants with school information, which the INEP read-me says was obtained by linking to the 2025 School Census; it may not represent all participants.

**Spread by area:** score dispersion differs by area (standard deviation: CN 77, CH 85, LC 73, MT 126), so gaps measured in points are not directly comparable across areas.

**School information.** According to the INEP read-me (pp. 7–8), the school code comes from the 2025 School Census, linked by CPF, restricted to enrollments in final-year stages of high school (3rd/4th year, integrated technical, "normal", non-graded and EJA). Participants with no such match have no school information (about 61% of present participants). Public vs. private comparisons therefore apply only to participants matched to a final-year enrollment, not to all participants. For students with more than one enrollment, INEP gave priority to public-network, regular-education, longer-hours and same-municipality enrollments. INEP advises against using these data to rank schools; this project reports only aggregates by state, region and school type. The questionnaire lives in a file that cannot be linked, so socioeconomic status cannot be controlled for.

**Geography.** Municipality of residence was removed from the 2025 microdata, so state means refer to the state where the exam was taken (`SG_UF_PROVA`).

**Removed records.** INEP removed participants of a few exam versions with very few takers, to prevent identification.

In [ ]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
df.to_parquet("../data/processed/results_2025_selected_columns.parquet")
com_escola.to_parquet("../data/processed/results_2025_with_school.parquet")

In [ ]:
por_uf_lc = pivo(com_escola, "SG_UF_ESC", "LC")

for corte in [500, 1000, 2000]:
    selecao = por_uf_lc[por_uf_lc["n_priv"] >= corte].sort_values("gap")
    print(
        "corte", corte, "| estados:", len(selecao),
        "| menor gap:", selecao.index[0], round(selecao["gap"].iloc[0], 1),
        "| maior gap:", selecao.index[-1], round(selecao["gap"].iloc[-1], 1),
    )

In [ ]:
nacional = {}
for area in areas:
    notas = df.loc[df[f"TP_PRESENCA_{area}"] == 1, f"NU_NOTA_{area}"]
    nacional[area] = (notas.mean(), notas.std())

z = pd.DataFrame(index=medias_uf.index)
for area in areas:
    media, desvio = nacional[area]
    z[area] = (medias_uf[area] - media) / desvio

z["amplitude"] = z[areas].max(axis=1) - z[areas].min(axis=1)
print(z.round(2).sort_values("amplitude").to_string())

In [ ]:
guia = list(pasta.glob("*/Entenda_a_Sua_Nota_no_Enem_Guia_do_Participante.pdf"))
leitor = PdfReader(guia[0])
for i, pagina in enumerate(leitor.pages):
    texto = pagina.extract_text() or ""
    for linha in texto.split("\n"):
        if "escala" in linha.lower() or "compar" in linha.lower():
            print(i + 1, "|", linha)

### Robustness checks

**Minimum private-school sample per state (LC gap):** thresholds of 500, 1,000 and 2,000 private-school students leave 26, 23 and 22 states; the states with the smallest (SP) and largest (AM) public–private gaps do not change. The MT ranking was not tested.

**State profile across areas (z-scores):** state means were standardized against the national mean and the individual-level standard deviation of each area. Scores are consistently above or below the national mean across the four areas; the largest spread between areas within a state is about 0.2 standard deviations (AP). "Balanced" here means similar across areas, not high.

In [ ]:
import seaborn as sns

ordem = z[areas].mean(axis=1).sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(7, 10))
sns.heatmap(
    z.loc[ordem, areas],
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0,
    cbar_kws={"label": "Std. deviations from national mean"},
    ax=ax,
)
ax.set_title("State mean score by area, relative to the national mean\n(participants present, state of exam)")
ax.set_xlabel("Area")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig("../images/heatmap_state_area_z.png", dpi=150)
plt.show()

In [ ]:
lc = df.loc[df["TP_PRESENCA_LC"] == 1, "NU_NOTA_LC"]
media, mediana = lc.mean(), lc.median()
print("LC média:", round(media, 1), "| mediana:", round(mediana, 1))

lc_escola = com_escola[com_escola["TP_PRESENCA_LC"] == 1].copy()
lc_escola["School type"] = lc_escola["REDE"].map({"Pública": "Public", "Privada": "Private"})

fig, eixos = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(lc, bins=60, ax=eixos[0])
eixos[0].axvline(media, linestyle="--", color="black", label=f"Mean {media:.1f}")
eixos[0].axvline(mediana, linestyle=":", color="red", label=f"Median {mediana:.1f}")
eixos[0].set_title("LC scores, all participants present")
eixos[0].set_xlabel("Score")
eixos[0].legend()

sns.histplot(
    data=lc_escola, x="NU_NOTA_LC", hue="School type",
    bins=60, stat="density", common_norm=False, element="step", ax=eixos[1],
)
eixos[1].set_title("LC scores by school type\n(only participants with school information)")
eixos[1].set_xlabel("Score")

plt.tight_layout()
fig.savefig("../images/lc_distributions.png", dpi=150)
plt.show()

In [28]:

for area in areas:
    notas = df.loc[df[f"TP_PRESENCA_{area}"] == 1, f"NU_NOTA_{area}"]
    print(
        area,
        "| notas == 0:", (notas == 0).sum(),
        "| menor nota > 0:", notas[notas > 0].min(),
        "| média sem zeros:", round(notas[notas > 0].mean(), 2),
        "| média com zeros:", round(notas.mean(), 2),
    )
print()


pub = lc_escola.loc[lc_escola["School type"] == "Public", "NU_NOTA_LC"]
priv = lc_escola.loc[lc_escola["School type"] == "Private", "NU_NOTA_LC"]
print("Percentis 10/25/50/75/90, pública: ", pub.quantile([.1, .25, .5, .75, .9]).round(0).tolist())
print("Percentis 10/25/50/75/90, privada: ", priv.quantile([.1, .25, .5, .75, .9]).round(0).tolist())
print("% de alunos da pública acima da mediana da privada:", round((pub > priv.median()).mean() * 100, 1))
print("% de alunos da privada abaixo da mediana da pública:", round((priv < pub.median()).mean() * 100, 1))

CN | notas == 0: 775 | menor nota > 0: 308.6 | média sem zeros: 500.09 | média com zeros: 499.97
CH | notas == 0: 9087 | menor nota > 0: 320.8 | média sem zeros: 512.54 | média com zeros: 511.19
LC | notas == 0: 2361 | menor nota > 0: 309.2 | média sem zeros: 532.49 | média com zeros: 532.12
MT | notas == 0: 893 | menor nota > 0: 312.6 | média sem zeros: 520.12 | média com zeros: 519.98

Percentis 10/25/50/75/90, pública:  [414.0, 470.0, 520.0, 561.0, 595.0]
Percentis 10/25/50/75/90, privada:  [501.0, 543.0, 584.0, 619.0, 649.0]
% de alunos da pública acima da mediana da privada: 14.1
% de alunos da privada abaixo da mediana da pública: 15.4


### Comparing areas

According to the INEP participant guide (pp. 11, 25), each knowledge area has its own scale, so raw scores of different areas are not directly comparable. In this project, comparisons are made *within* an area (between states, regions and school types). Standardized measures (z-scores, differences in standard deviations) are used only to describe relative position within each area, not to claim that areas measure the same thing.

### Distribution notes (LC)

LC scores are left-skewed (mean 532.1, median 538.8). Public and private school distributions overlap substantially: the mean gap (65.6 points, about 0.9 standard deviations) does not mean that every private-school student outscores every public-school student. A small group of LC scores equal to zero was found and is being investigated.